In [1]:
import json
import joblib
import pandas as pd
import wandb
from dotenv import load_dotenv
from tune import run_study
from train import run_cv, train_final
from features import PIPELINE_STEPS

load_dotenv()

df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

/Users/aleksandragakin/projects/study/spaceship_titanic/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RANDOM_SEED = 42


## Tune

In [ ]:
CV_FOLDS = 10
N_TRIALS = 100
STUDY_NAME = "catboost-v3"

study = run_study(df, n_trials=N_TRIALS, study_name=STUDY_NAME, cv_folds=CV_FOLDS, random_seed=RANDOM_SEED)

  0%|          | 0/1 [00:00<?, ?it/s]wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
Best trial: 0. Best value: 0.808466: 100%|██████████| 1/1 [02:21<00:00, 141.56s/it]


Best CV:     0.80847
Best params: {'iterations': 2185, 'learning_rate': 0.22648248189516842, 'depth': 8, 'l2_leaf_reg': 3.968793330444372, 'bagging_temperature': 0.15601864044243652}
Saved best_params.json


## Train

In [ ]:
with open("best_params.json") as f:
    params = json.load(f)

mean, std = run_cv(params, df, cv_folds=CV_FOLDS, random_seed=RANDOM_SEED)
print(f"CV: {mean:.5f} ± {std:.5f}")

In [ ]:
pipeline = train_final(df, params, random_seed=RANDOM_SEED)

wandb.init(project="spaceship-titanic", job_type="train", config={
    **params,
    "model": "catboost",
    "cv_folds": CV_FOLDS,
    "pipeline_steps": PIPELINE_STEPS,
})
wandb.log({"cv_mean": mean, "cv_std": std})
wandb.finish()

## Predict

In [5]:
pipeline = joblib.load("pipeline.pkl")

passenger_ids = test_df["PassengerId"].copy()
preds = pipeline.predict(test_df).astype(bool)

submission = pd.DataFrame({"PassengerId": passenger_ids, "Transported": preds})
submission.to_csv("submission.csv", index=False)

n_transported = preds.sum()
wandb.init(
    project="spaceship-titanic",
    job_type="predict",
    config=pipeline.named_steps["model"].get_params(),
)
wandb.log({
    "n_rows": len(submission),
    "n_transported": int(n_transported),
    "transport_rate": round(n_transported / len(submission), 4),
})
wandb.finish()

print(f"Saved submission.csv ({len(submission)} rows)")
print(f"Transported: {n_transported} / {len(submission)} ({n_transported / len(submission):.1%})")

n_rows,▁
n_transported,▁
transport_rate,▁
n_rows,4277
n_transported,2137
transport_rate,0.4996


Saved submission.csv (4277 rows)
Transported: 2137 / 4277 (50.0%)
